# In-class recall replay

Runs **`inclass_simulation.py`** as a subprocess with `cwd = ok1/`.

You **must** pass either **`--class-id N`** or **`--all-classes`** on the CLI (the script exits otherwise).

Outputs: `outputs/<dataset>/inclass/class<id>/<pixels|pca{d}>/Radius<R>/{result.csv, recall_plot.png}`.

### Setup
- Use the **`ok1/.venv`** kernel.
- Run from **`ok1/`** or **`ok1/notebooks/`** so repo discovery works.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents[:4]]
    for d in candidates:
        if (d / "inclass_simulation.py").is_file():
            return d
    raise RuntimeError(
        "Could not find inclass_simulation.py. cd into ok1/ or ok1/notebooks/."
    )


REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def run_inclass(*argv: str) -> None:
    script = REPO_ROOT / "inclass_simulation.py"
    cmd = [sys.executable, str(script), *argv]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)

### Example: single CIFAR-10 class, small M (quick)

Adjust `--mem-R`, `--pca-dim`, or use `--no-pca` like the README shell examples.

In [ ]:
run_inclass(
    "--dataset",
    "cifar10",
    "--class-id",
    "3",
    "--pca-dim",
    "32",
    "--M-min",
    "15",
    "--M-max",
    "80",
    "--n-trials",
    "2",
    "--max-steps",
    "8",
    "--mem-R",
    "2",
    "--device",
    "cpu",
)

### Optional: all 10 classes (longer)

Remove the block comment when you want a full sweep instead of `--class-id`.

In [ ]:
# run_inclass(
#     "--dataset", "cifar10",
#     "--all-classes",
#     "--no-pca",
#     "--M-min", "10",
#     "--M-max", "100",
#     "--n-trials", "2",
#     "--device", "cpu",
# )

### Inspect CSV / plot paths for last run parameters

Match **`class_id`** and PCA settings below to the **`run_inclass(...)` cell** above.

In [ ]:
from argparse import Namespace

import pandas as pd
from IPython.display import display

# Mirror the argparse flags from the "Example" cell above:
args = Namespace(
    output_dir="outputs",
    dataset="cifar10",
    mem_R=2.0,
    no_pca=False,
    d=50,
    pca_dim=None,
)
# If you ran with --pca-dim 32, set explicitly (ignored when no_pca=True):
args.pca_dim = 32

class_id = 3

from icml_hyp.cli.inclass_simulation import output_paths

csv_path, plot_path = output_paths(args, class_id)
full_csv = REPO_ROOT / csv_path
print("CSV:", full_csv.resolve())
print("Plot:", (REPO_ROOT / plot_path).resolve())
if full_csv.is_file():
    display(pd.read_csv(full_csv).head(20))